# Fine-Tune Qwen 1.5B Instruct (ChatML JSONL)
Make sure your runtime is set to **GPU** (**Runtime** → **Change runtime type** → **T4 GPU**).

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
from google.colab import files

uploaded = files.upload()
dataset_filename = list(uploaded.keys())[0]
print(f"Uploaded: {dataset_filename}")

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

dataset = load_dataset("json", data_files=dataset_filename, split="train")

def formatting_prompts_func(examples):
    if "messages" in examples or "conversations" in examples:
        col = "messages" if "messages" in examples else "conversations"
        convos = examples[col]
        texts = [
            tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
            for convo in convos
        ]
        return {"text": texts}
    elif "text" in examples:
        return {"text": examples["text"]}
    else:
        raise ValueError(f"Unsupported columns found: {list(examples.keys())}")

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Loaded {len(dataset)} records.")
print("Sample formatted record:\n", dataset[0]["text"])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Replace this with a test prompt matching your dataset!"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7)
print(tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0])

In [ ]:
model.save_pretrained("qwen_1.5b_lora")
tokenizer.save_pretrained("qwen_1.5b_lora")
!zip -r qwen_1.5b_lora.zip qwen_1.5b_lora
files.download("qwen_1.5b_lora.zip")